In [35]:
# In this notebook I will merge the proceedings_final data with application_data_final using patent number as a key

import re
import pandas as pd
import numpy as np

application_data_final = pd.read_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/merged_data/application_data_final.csv")
proceedings_final = pd.read_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/merged_data/proceedings_final.csv")


In [36]:
# I merge two data sets using the examiner art unit column for grant propensity art unit
au_lookup = (application_data_final[["examiner_art_unit","grant_propensity_art_unit"]]
             .dropna(subset=["examiner_art_unit"])
             .drop_duplicates(subset=["examiner_art_unit"])
             )

merged_data = proceedings_final.merge(au_lookup, on="examiner_art_unit", how="left")



In [37]:
# I merge two data sets using the examiner art unit column for leniecny art unit
au_lookup = (application_data_final[["examiner_art_unit","leniency_art_unit"]]
             .dropna(subset=["examiner_art_unit"])
             .drop_duplicates(subset=["examiner_art_unit"])
             )

merged_data = merged_data.merge(au_lookup, on="examiner_art_unit", how="left")

In [38]:
# I fix tech center and art unit Dtype
merged_data["examiner_tech_center_number"]= pd.to_numeric(merged_data["examiner_tech_center_number"], errors="coerce").astype("Int64")
merged_data["examiner_art_unit"]= pd.to_numeric(merged_data["examiner_art_unit"], errors="coerce").astype("Int64")

In [39]:
# I count the rows that have multiple petitioner names in the string
COL = "petitioner"
s = merged_data[COL].fillna("").astype(str)

# et al
et_pattern = re.compile(r"\bet\.?\s*al\.?\b", flags=re.IGNORECASE)
merged_data["et_al_flag"] = s.str.contains(et_pattern, na=False)

# entity suffix pattern (extend as needed)
entity_suffix = r"(?:inc\.?|corp\.?|co\.?|ltd\.?|llc|gmbh|s\.?p\.?a\.?|s\.?a\.?|ag|bv|nv|plc|pte\.?|kk|oy|ab)\b"

# 1) Strong delimiter: semicolon (rarely inside a single name)
has_semicolon = s.str.contains(r";", regex=True)

# 2) " and " delimiter: require TWO entity suffixes (guards against "Research and Development")
# Count how many suffixes appear in the string; if >=2 and contains ' and ', treat as multi-name
suffix_count = s.str.count(entity_suffix, flags=re.IGNORECASE)
has_and_two_suffixes = s.str.contains(r"\band\b", flags=re.IGNORECASE, regex=True) & (suffix_count >= 2)

# 3) Slash delimiter: also require TWO entity suffixes
has_slash_two_suffixes = s.str.contains(r"/", regex=True) & (suffix_count >= 2)

# 4) Comma-based lists: ONLY if there are >=2 entity suffixes (prevents "Guardant Health, Inc.")
# This catches strings like "TikTok Inc., Bytedance Ltd, Bytedance Pte. Ltd."
has_comma_two_suffixes = s.str.contains(r",", regex=True) & (suffix_count >= 2)

merged_data["multi_name_flag"] = has_semicolon | has_and_two_suffixes | has_slash_two_suffixes | has_comma_two_suffixes
merged_data["multi_petitioner_flag"] = merged_data["et_al_flag"] | merged_data["multi_name_flag"]

print("et al share:", merged_data["et_al_flag"].mean())
print("multi-name share (non-et al):", (merged_data["multi_name_flag"] & ~merged_data["et_al_flag"]).mean())
print("combined multi-petitioner share:", merged_data["multi_petitioner_flag"].mean())
print("How many rows would be 'explodable' (non-et al):",
      int((merged_data["multi_name_flag"] & ~merged_data["et_al_flag"]).sum()))

et al share: 0.42988621151271755
multi-name share (non-et al): 0.05204149933065596
combined multi-petitioner share: 0.4819277108433735
How many rows would be 'explodable' (non-et al): 933


In [40]:
# The examples of rows that have multiple petitioner names
COL = "petitioner"
examples = merged_data.loc[
    merged_data["multi_name_flag"] & ~merged_data["et_al_flag"],
    COL
].drop_duplicates().head(50)

print(examples.to_string(index=False))


                               Cisco Systems, Inc.
SHENZHEN QIANFENYI INTELLIGENT TECHNOLOGY CO., ...
             Bonerge Lifescience (Hunan) Co., Ltd.
                    BOE Technology Group Co., Ltd.
                    BOE TECHNOLOGY GROUP CO., LTD.
                         Jesco Lighting Group, LLC
SHENZHEN RONGLIDA TECHNOLOGY CO. LTD. d/b/a Shu...
TikTok Inc. and Bytedance Ltd, Bytedance Pte. L...
                 Caihong Display Devices Co., Ltd.
                            Nissan Motor Co., Ltd.
                 Caihong Display Devices, Co., Ltd
                                 L'Oreal USA, Inc.
    Suzhou Mojawa Intelligent Electronic Co., Ltd.
                           Home Depot U.S.A., Inc.
                    Zhuhai CosMX Battery Co., Ltd.
Kingston Technology Company, Inc., Kingston Tec...
Yealink (USA) Network Technology Co., Ltd. and ...
Kangxi Communications Technologies (Shanghai) C...
                               Ajinomoto Co., Inc.
            Normshield, Inc. d/

In [41]:
COL = "respondent"
s = merged_data[COL].fillna("").astype(str)

# et al
et_pattern = re.compile(r"\bet\.?\s*al\.?\b", flags=re.IGNORECASE)
merged_data["et_al_flag2"] = s.str.contains(et_pattern, na=False)

# entity suffix pattern (extend as needed)
entity_suffix = r"(?:inc\.?|corp\.?|co\.?|ltd\.?|llc|gmbh|s\.?p\.?a\.?|s\.?a\.?|ag|bv|nv|plc|pte\.?|kk|oy|ab)\b"

# 1) Strong delimiter: semicolon (rarely inside a single name)
has_semicolon2 = s.str.contains(r";", regex=True)

# 2) " and " delimiter: require TWO entity suffixes (guards against "Research and Development")
# Count how many suffixes appear in the string; if >=2 and contains ' and ', treat as multi-name
suffix_count = s.str.count(entity_suffix, flags=re.IGNORECASE)
has_and_two_suffixes2 = s.str.contains(r"\band\b", flags=re.IGNORECASE, regex=True) & (suffix_count >= 2)

# 3) Slash delimiter: also require TWO entity suffixes
has_slash_two_suffixes2 = s.str.contains(r"/", regex=True) & (suffix_count >= 2)

# 4) Comma-based lists: ONLY if there are >=2 entity suffixes (prevents "Guardant Health, Inc.")
# This catches strings like "TikTok Inc., Bytedance Ltd, Bytedance Pte. Ltd."
has_comma_two_suffixes2 = s.str.contains(r",", regex=True) & (suffix_count >= 2)

merged_data["multi_name_flag2"] = has_semicolon2 | has_and_two_suffixes2 | has_slash_two_suffixes2 | has_comma_two_suffixes2
merged_data["multi_petitioner_flag2"] = merged_data["et_al_flag2"] | merged_data["multi_name_flag2"]

print("et al share:", merged_data["et_al_flag2"].mean())
print("multi-name share (non-et al):", (merged_data["multi_name_flag2"] & ~merged_data["et_al_flag2"]).mean())
print("combined multi-petitioner share:", merged_data["multi_petitioner_flag2"].mean())
print("How many rows would be 'explodable' (non-et al):",
      int((merged_data["multi_name_flag2"] & ~merged_data["et_al_flag2"]).sum()))

et al share: 0.1746987951807229
multi-name share (non-et al): 0.02744310575635877
combined multi-petitioner share: 0.20214190093708165
How many rows would be 'explodable' (non-et al): 492


In [42]:
# The examples of rows that have multiple respondent names
COL = "respondent"
examples = merged_data.loc[
    merged_data["multi_name_flag2"] & ~merged_data["et_al_flag2"],
    COL
].drop_duplicates().head(50)

print(examples.to_string(index=False))

                              LG Display Co., Ltd.
          Nanjing Nutrabuilding Bio-Tech Co., Ltd.
                         Samsung Display Co., Ltd.
                    Force MOS Technology Co., Ltd.
            Guangzhou Haoyang Electronic Co., Ltd.
                          Shenzhen Shokz Co., Ltd.
                         Atossa Therapeutics, Inc.
                                   AbTis Co., Ltd.
              Kyocera Senco Industrial Tools, Inc.
                            Neapco Components, LLC
                              SK nexilis Co., Ltd.
                         Unverferth Mfg. Co., Inc.
                    Bunge Loders Croklaan USA, LLC
                        Largan Precision Co., Ltd.
                      Hanshow Technology Co., Ltd.
                   Everlight Electronics Co., Ltd.
                                      Lumenco, LLC
                       Orange Electronic Co., Ltd.
                              Throughtek Co., Ltd.
               Shenzhen Carku T

In [43]:
merged_data["petitioner"].value_counts(dropna=False)

petitioner
Apple Inc.                                  819
Samsung Electronics Co., Ltd. et al.        588
Google LLC                                  322
Microsoft Corporation                       220
Intel Corporation                           187
                                           ... 
MAHLE FILTER SYSTEMS NORTH AMERICA, INC.      1
Athenahealth, Inc. et al.                     1
Kolbe & Kolbe Millwork Co., Inc.              1
u-blox AG et al.                              1
MACAUTO U.S.A.                                1
Name: count, Length: 4392, dtype: int64

In [44]:
merged_data.sample(10)

,patent_number,grant_date,examiner_tech_center_number,examiner_art_unit,inventor_name,trial_number,trial_status_category,petition_filing_date,institution_decision_date,termination_date,...,petitioner,trial_type,grant_propensity_art_unit,leniency_art_unit,et_al_flag,multi_name_flag,multi_petitioner_flag,et_al_flag2,multi_name_flag2,multi_petitioner_flag2
8087,8806957,2014-08-19,2800,2856,Peter Schmidt Laursen et al,IPR2019-01640,Final Written Decision,2019-09-23,2020-04-08,2021-04-01,...,Axioma Metering UAB,IPR,0.808223,0.808223,False,False,False,True,False,True
14603,7525484,2009-04-28,3600,3662,Dennis J. Dupray et al,IPR2015-01697,Terminated-Settled,2015-08-12,2016-02-17,2016-04-08,...,"Apple, Inc.",IPR,0.825421,0.835982,False,False,False,False,False,False
7103,7310304,2007-12-18,2600,2616,Apurva N. Mody et al,IPR2020-01178,Terminated-Settled,2020-06-23,NaN,2020-11-17,...,Hewlett Packard Enterprise Company,IPR,0.778735,0.790864,False,False,False,False,False,False
3490,7440559,2008-10-21,2600,2614,Ahti Muhonen et al,IPR2023-00630,Final Written Decision - Appealed,2023-02-23,2023-10-03,2024-10-02,...,"Netflix, Inc. et al.",IPR,0.697343,0.697343,True,False,True,True,False,True
15683,7523072,2009-04-21,3600,3621,Mark J. Stefik et al,IPR2015-00443,Institution Denied,2014-12-22,2015-07-09,2015-07-09,...,Apple Inc.,IPR,0.420029,0.420029,False,False,False,True,False,True
4726,7917680,2011-03-29,2100,2111,Kevin Locker,IPR2022-00688,Institution Denied,2022-03-14,2022-09-14,2022-09-14,...,Realtek Semiconductor Corp.,IPR,0.802857,0.819913,False,False,False,False,False,False
2254,8567988,2013-10-29,2800,2875,Rene Peter Helbing,IPR2024-00542,Terminated-Settled,2024-03-14,NaN,2024-07-24,...,Nichia Corporation,IPR,0.769398,0.784572,False,False,False,False,False,False
16001,8300863,2012-10-30,2800,2815,Ove Knudsen et al,IPR2015-00103,Final Written Decision,2014-10-21,2015-04-29,2015-10-07,...,GN RESOUND AS,IPR,0.772066,0.779814,False,False,False,False,False,False
5670,8977797,2015-03-10,2100,2111,William W. Y. Chu,IPR2021-01105,Institution Denied,2021-06-25,2022-01-05,2022-01-05,...,Intel Corporation et al.,IPR,0.802857,0.819913,True,False,True,False,False,False
9460,8478871,2013-07-02,2400,2445,Gerald Gutt et al,IPR2018-01823,Terminated-Settled,2018-09-28,2019-04-17,2019-07-08,...,"ipDataTel, LLC",IPR,0.701941,0.720579,False,False,False,False,False,False


In [45]:
# I remove the trade name of the firms and keep only the legal name
DBA_RE = re.compile(r"\b(?:d\s*/\s*b\s*/\s*a|dba|doing business as)\b", flags=re.IGNORECASE)

def drop_dba_tail(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return s

    s = str(s)

    # remove parenthetical d/b/a parts like "(d/b/a Something)"
    s = re.sub(r"\(\s*(?:d\s*/\s*b\s*/\s*a|dba|doing business as)\b.*?\)", "", s, flags=re.IGNORECASE).strip()

    # remove inline d/b/a tail like ", Inc. d/b/a Mockingbird"
    s = DBA_RE.split(s)[0].strip()

    return s

merged_data2 = merged_data.copy()
merged_data2["respondent"] = merged_data["respondent"].apply(drop_dba_tail)
merged_data2["petitioner"] = merged_data["petitioner"].apply(drop_dba_tail)

In [46]:
# I remove et al. suffix
suffixes = ["et al."
]

suffix_pattern = r"(?:\s*,?\s*(?:" + "|".join(map(re.escape, suffixes)) + r")\.?\s*)$"

def strip_legal_suffixes(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)

    # iterative in case of "Co., Ltd." etc
    while True:
        new = re.sub(suffix_pattern, "", s, flags=re.IGNORECASE).rstrip(" ,.")
        if new == s:
            break
        s = new
    return s

merged_data2["respondent"] = merged_data2["respondent"].apply(strip_legal_suffixes)
merged_data2["petitioner"] = merged_data2["petitioner"].apply(strip_legal_suffixes)

In [47]:
# I normalize the names in the entries such as "SAMSUNG ELECTRONICS CO., LTD. -> Samsung Electronics Co., Ltd"
def fix_all_caps(name: str) -> str:
    if name is None:
        return name
    s = str(name).strip()

    # If there are no letters, or it's not all-caps, leave it
    letters = re.findall(r"[A-Za-z]", s)
    if not letters:
        return s

    # "ALL CAPS" test: every letter is uppercase
    if all(ch.isupper() for ch in letters):
        # Title-case it
        s2 = s.title()

        # Optional: keep certain short tokens uppercase (e.g., LLC, USA, GmbH) if they remain
        # and keep tokens with digits/symbols like "3M" as-is
        tokens = []
        for tok in s2.split():
            raw = tok.strip(",.")
            if any(c.isdigit() for c in raw) or "&" in raw:
                tokens.append(tok.upper())  # e.g., "3m"->"3M", "At&t"->"AT&T"
            else:
                tokens.append(tok)
        return " ".join(tokens)

    return s

merged_data2["respondent"] = merged_data2["respondent"].apply(fix_all_caps)
merged_data2["petitioner"] = merged_data2["petitioner"].apply(fix_all_caps)

In [48]:
merged_data2.head(10)

,patent_number,grant_date,examiner_tech_center_number,examiner_art_unit,inventor_name,trial_number,trial_status_category,petition_filing_date,institution_decision_date,termination_date,...,petitioner,trial_type,grant_propensity_art_unit,leniency_art_unit,et_al_flag,multi_name_flag,multi_petitioner_flag,et_al_flag2,multi_name_flag2,multi_petitioner_flag2
0,9772193,2017-09-26,2600,2648,Ehud MENDELSON,IPR2026-00156,Pending,2025-12-31,NaN,NaN,...,Google LLC,IPR,0.832716,0.852327,False,False,False,False,False,False
1,12231426,2025-02-18,2400,2495,Jason Crabtree et al,IPR2026-00184,Pending,2025-12-30,NaN,NaN,...,Microsoft Corporation,IPR,0.783444,0.813932,False,False,False,False,False,False
2,12218934,2025-02-04,2400,2495,Jason Crabtree et al,IPR2026-00182,Pending,2025-12-30,NaN,NaN,...,Microsoft Corporation,IPR,0.783444,0.813932,False,False,False,False,False,False
3,10991097,2021-04-27,2600,2666,Stephen Yip et al,IPR2026-00185,Pending,2025-12-30,NaN,NaN,...,"Guardant Health, Inc",IPR,0.846213,0.859270,False,False,False,False,False,False
4,9456053,2016-09-27,2400,2454,Christopher Newton et al,IPR2026-00190,Pending,2025-12-29,NaN,NaN,...,Microsoft Corporation,IPR,0.792499,0.804758,False,False,False,False,False,False
5,10701173,2020-06-30,2400,2458,Christopher Newton et al,IPR2026-00180,Pending,2025-12-24,NaN,NaN,...,Microsoft Corporation,IPR,0.691235,0.713811,False,False,False,False,False,False
6,11058757,2021-07-13,1600,1645,Bruce D. FORREST et al,IPR2026-00189,Pending,2025-12-23,NaN,NaN,...,Pfizer Inc,IPR,0.631438,0.643398,False,False,False,False,False,False
7,11806526,2023-11-07,3700,3792,Darrell Wagner et al,IPR2026-00091,Pending,2025-12-18,NaN,NaN,...,"Nyxoah, Inc",IPR,0.746274,0.791046,True,False,True,False,False,False
8,10898709,2021-01-26,3700,3792,Darrell Wagner et al,IPR2026-00090,Pending,2025-12-18,NaN,NaN,...,"Nyxoah, Inc",IPR,0.746274,0.791046,True,False,True,False,False,False
9,11850424,2023-12-26,3700,3792,Darrell Wagner et al,IPR2026-00092,Pending,2025-12-18,NaN,NaN,...,"Nyxoah, Inc",IPR,0.746274,0.791046,True,False,True,False,False,False


In [49]:
merged_data2 = merged_data2.drop(columns={"trial_type"})

In [50]:
# I check for exactly the same rows
exact_dup_mask = merged_data2.duplicated(keep=False)
print("Exact duplicate rows:", exact_dup_mask.sum())


Exact duplicate rows: 0


In [51]:
# I check for duplicates in patent number.
dup_mask = merged_data2.duplicated(subset=["patent_number"], keep=False)
print("Duplicate rows (by patent_number):", dup_mask.sum())
print("Share of rows duplicated:", dup_mask.mean())


Duplicate rows (by patent_number): 9558
Share of rows duplicated: 0.5331325301204819


In [52]:
# I check for duplicates in patent number.
dup_mask2 = merged_data2.duplicated(subset=["trial_number"], keep=False)
print("Duplicate rows (by trial_number):", dup_mask2.sum())
print("Share of rows duplicated:", dup_mask2.mean())

Duplicate rows (by trial_number): 0
Share of rows duplicated: 0.0


In [53]:
n_unique = merged_data2["patent_number"].nunique(dropna=True)
print("Unique patents:", n_unique)

Unique patents: 11732


In [54]:
# I check for duplicates in patent number.
dup_mask3 = merged_data2.duplicated(subset=["respondent"], keep=False)
print("Duplicate rows (by respondent):", dup_mask3.sum())
print("Share of rows duplicated:", dup_mask3.mean())

Duplicate rows (by respondent): 16156
Share of rows duplicated: 0.9011601963409193


In [55]:
merged_data2["respondent"] = merged_data2["respondent"].replace("nan", pd.NA)
merged_data2["respondent"].value_counts(dropna=False)

respondent
<NA>                            184
Intellectual Ventures II LLC    115
Uniloc 2017 LLC                 107
Rovi Guides, Inc                 98
Maxell, Ltd                      78
                               ... 
SilcoTek Corporation              1
Ahern Rentals, Inc                1
Shipping and Transit, LLC         1
Sanho Corporation                 1
Bergstrom, Inc                    1
Name: count, Length: 4290, dtype: int64

In [56]:
dup_mask4 = merged_data2.duplicated(subset=["petitioner"], keep=False)
print("Duplicate rows (by petitioner):", dup_mask4.sum())
print("Share of rows duplicated:", dup_mask4.mean())

Duplicate rows (by petitioner): 16361
Share of rows duplicated: 0.912594823739402


In [57]:
merged_data2["petitioner"] = merged_data2["petitioner"].replace("nan", pd.NA)
merged_data2["petitioner"].value_counts(dropna=True)

petitioner
Apple Inc                                            926
Samsung Electronics Co., Ltd                         866
Google LLC                                           436
Microsoft Corporation                                286
Intel Corporation                                    266
                                                    ... 
TCL China Star OptoElectonics Technology Co., Ltd      1
Prometric Inc                                          1
CWGC LA Inc                                            1
Grit Energy Solutions, LLC                             1
Macauto U.S.A                                          1
Name: count, Length: 3588, dtype: int64

In [58]:
# I download the petitioner names for firm matching later
merged_data2["petitioner"].to_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/merged_data/petitioner_name.csv", index = False)

In [59]:
# I download the respondent names for firm matching later
merged_data2["respondent"].to_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/merged_data/respondent_name.csv", index = False)

In [60]:
merged_data2.to_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/orbis_data/merged_final.csv", index = False)